## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e9

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E9" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E9/smoke" if SMOKE else "E9"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e9.build_initial_spec(SMOKE)

## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e9.E9_MODULES)
e9.verify_spec_immutable(OUT, SPEC, e9.E9_MODULES)

## 3. Leakage test

In [ ]:
PANEL = e9.load_panel(SPEC, CACHE)
leaks = e9.run_leakage_e9(PANEL, iso="CAN")
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
assert ldf["pass"].all(), ldf[~ldf["pass"]]

## 4. Load data

In [ ]:
print("countries:", SPEC["countries"])
print(PANEL["frames"]["USA"].index[[0, -1]])

## 5. Reference model walk-forward + reproduction check

In [ ]:
ref_df, oecd_ok, usa_warn = e9.run_reference_reproduction(
    PANEL, e9.PRIMARY_H, e9.ref_check_isos(SMOKE)
)
print(ref_df.to_string(index=False))
if usa_warn:
    print("NOTE: USA uses CPIAUCSL (plan); crosscountry reference AUC used OECD USA.")
if not oecd_ok:
    raise RuntimeError("OECD reference AUC check failed (see reference_check.csv)")

## 6. Losses

## 7. Features

## 8. Loss predictor walk-forward

## 9. Decision

In [ ]:
RESULT = e9.run_full_pipeline(SPEC, OUT, FIG, CACHE, data=PANEL)

## 10. Secondary

## 11. Figures and tables

In [ ]:
print(pd.read_csv(OUT / "decision.csv").to_string(index=False))
for name in (
    "reference_check",
    "secondary_pooled",
    "secondary_families_holm",
    "secondary_event",
    "secondary_us",
):
    path = OUT / f"{name}.csv"
    if path.exists():
        print(name)
        print(pd.read_csv(path).to_string(index=False))
case = OUT / "secondary_case_2021_2023.csv"
if case.exists():
    cdf = pd.read_csv(case)
    print("secondary_case_2021_2023", cdf.groupby("iso").size().to_dict())